In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(

    embeddings=embedding_model,
    connection=connection_url,
    collection_name="integrated_university_rules",
    use_jsonb=True # 메타데이터를 JSONB형태로 저장
)

In [6]:
from langchain_core.documents import Document

# 1차 등록 대상 문서 (2026 규정 및 장학 규정)
initial_documents = [
    Document(
        page_content="[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.",
        metadata={"source": "rules_library.txt", "category": "library", "year": 2026}
    ),
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.",
        metadata={"source": "rules_scholarship.txt", "category": "scholarship", "year": 2026}
    )
]

In [7]:
from langchain_classic.indexes import SQLRecordManager, index

record_manager = SQLRecordManager(
    namespace="integrated_university_rules",
    db_url=connection_url,
)

record_manager.create_schema() # 추적용 테이블 

In [10]:
first_result = index(
    docs_source=initial_documents,
    record_manager=record_manager,
    vector_store=vector_store,
    source_id_key="source",
    cleanup="incremental"
)

"""
none : 증분 관리 X
incremental : 중복된 레코드가 있으면 삭제하고 새로 추가
full : 전체를 삭제하고 다시 추가
"""

print(first_result)

{'num_added': 2, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}


In [11]:
updated_documents = [
    Document(
        page_content="[개정 도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.",
        metadata={"source": "rules_library.txt", "category": "library", "year": 2026}
    ),
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.",
        metadata={"source": "rules_scholarship.txt", "category": "scholarship", "year": 2026}
    )
]

second_result = index(
    docs_source=updated_documents,
    record_manager=record_manager,
    vector_store=vector_store,
    source_id_key="source",
    cleanup="incremental"
)

print(second_result)

{'num_added': 1, 'num_updated': 0, 'num_skipped': 1, 'num_deleted': 1}
